# FAISS Vectorstore Rebuild

Rebuild the FAISS index from the unified document chunks pipeline.

**Purpose:**
- Load `chunks/doc_chunks.json` (output from `document_processor.py`)
- Create embeddings with `sentence-transformers/all-MiniLM-L6-v2`
- Save fresh FAISS index to `vectordb/`
- Eliminate dependency on stale index

In [ ]:
%pip install sentence-transformers faiss-cpu langchain -q

In [ ]:
import json
from pathlib import Path

from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.schema import Document

print("✓ All imports successful")

In [ ]:
# Setup paths
BASE_DIR = Path("..").resolve()  # Project root
CHUNK_FILE = BASE_DIR / "chunks" / "doc_chunks.json"
VECTORDB_DIR = BASE_DIR / "vectordb"

print(f"Base directory: {BASE_DIR}")
print(f"Chunk file: {CHUNK_FILE}")
print(f"Vectorstore dir: {VECTORDB_DIR}")
print(f"\nChunk file exists: {CHUNK_FILE.exists()}")
print(f"Vectorstore dir exists: {VECTORDB_DIR.exists()}")

In [ ]:
# Load chunks from JSON
with open(CHUNK_FILE, "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"✓ Loaded {len(chunks)} chunks from {CHUNK_FILE.name}")
print(f"\nFirst chunk structure:")
if chunks:
    print(f"  page_content length: {len(chunks[0].get('page_content', ''))}")
    print(f"  metadata keys: {list(chunks[0].get('metadata', {}).keys())}")
    print(f"  sample content: {chunks[0]['page_content'][:200]}...")

In [ ]:
# Convert chunks to LangChain Document objects
documents = [
    Document(
        page_content=chunk["page_content"],
        metadata=chunk.get("metadata", {})
    )
    for chunk in chunks
]

print(f"✓ Converted {len(documents)} chunks to LangChain Documents")
print(f"\nFirst document:")
print(f"  Content length: {len(documents[0].page_content)} chars")
print(f"  Metadata: {documents[0].metadata}")

In [ ]:
# Initialize embeddings model
embedding_model = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model)

print(f"✓ Initialized embeddings: {embedding_model}")
print(f"  Embedding dimension: 384")

In [ ]:
# Build FAISS index from documents
print(f"Building FAISS index from {len(documents)} documents...")
print("This may take a moment...\n")

vectorstore = FAISS.from_documents(
    documents=documents,
    embedding=embeddings
)

print(f"✓ FAISS index created successfully!")
print(f"  Index type: {type(vectorstore).__name__}")
print(f"  Number of vectors: {len(documents)}")

In [ ]:
# Save FAISS index to disk
VECTORDB_DIR.mkdir(parents=True, exist_ok=True)
vectorstore.save_local(str(VECTORDB_DIR))

print(f"✓ FAISS index saved to: {VECTORDB_DIR.resolve()}")
print(f"\nIndex files created:")
for file in sorted(VECTORDB_DIR.glob("*")):
    size_mb = file.stat().st_size / (1024 * 1024)
    print(f"  - {file.name} ({size_mb:.2f} MB)")

In [ ]:
# Verify the index works with a test query
test_query = "What is machine learning?"

results = vectorstore.similarity_search(test_query, k=3)

print(f"✓ Test query successful: \"{test_query}\"")
print(f"\nRetrieved {len(results)} relevant chunks:")
for i, doc in enumerate(results, 1):
    print(f"\n{i}. Page {doc.metadata.get('page')}, Chunk {doc.metadata.get('page_chunk')}")
    print(f"   Words: {doc.metadata.get('word_count')}")
    print(f"   Content: {doc.page_content[:150]}...")

In [ ]:
# Final summary
print("=" * 60)
print("✅ FAISS REBUILD COMPLETE")
print("=" * 60)
print(f"\nPipeline steps completed:")
print(f"  1. ✓ Loaded {len(chunks)} chunks from doc_chunks.json")
print(f"  2. ✓ Converted to LangChain Documents")
print(f"  3. ✓ Initialized embeddings (sentence-transformers)")
print(f"  4. ✓ Built FAISS index from {len(documents)} documents")
print(f"  5. ✓ Saved to {VECTORDB_DIR.resolve()}")
print(f"  6. ✓ Verified retrieval with test query")
print(f"\n📁 Your system is now ready:")
print(f"  - Document processor: src/document_processor.py")
print(f"  - Chunks storage: chunks/doc_chunks.json")
print(f"  - Vector index: vectordb/")
print(f"  - Retriever: src/retriever.py")
print(f"  - Generator: src/generator.py")
print(f"  - Web app: app.py (Streamlit)")
print(f"\n🚀 Next: Run the Streamlit app!")
print(f"   Command: streamlit run app.py")